In [1]:
#pip install chess
import chess
import math
import random
import multiprocessing
from multiprocessing import Pool
from multiprocessing.dummy import Pool as ThreadPool

Select the child with the highest UCB1 score refers to the process of selecting the most promising child node in a Monte Carlo Tree Search (MCTS) algorithm using the Upper Confidence Bound 1 (UCB1) formula. This formula balances exploration (trying less-visited nodes to discover their potential) and exploitation (choosing nodes that have already shown good results)

UCB1 = (wins / visits) + c_param * sqrt((2 * log(parent_visits)) / visits)

Explanation of the components:
1. wins / visits: The average reward (or win rate) for the node. This represents the exploitation part, favoring nodes that have performed well so far.
2. c_param: A constant that controls the balance between exploration and exploitation. Higher values encourage exploration.
3. sqrt((2 * log(parent_visits)) / visits): The exploration term. It increases for nodes that have been visited less often, encouraging the algorithm to explore them.
The child node with the highest UCB1 score is selected because it represents the best trade-off between exploring new possibilities and exploiting known good results. This ensures that the search does not get stuck in local optima and explores the game tree effectively.

In [2]:
class GameState:
    def __init__(self, board=None):
        self.board = board if board is not None else chess.Board()
    
    def get_possible_moves(self):
        return list(self.board.legal_moves)
    
    def make_move(self, move):
        if move not in self.board.legal_moves:
            raise ValueError(f"Illegal move: {move.uci()}")
        new_board = self.board.copy()
        new_board.push(move)
        return GameState(new_board)
    
    def is_terminal(self):
        return self.board.is_game_over()
    
    def get_reward(self):
        # Reward: +1 (bot wins), -1 (bot loses), 0 (draw), intermediate: material eval
        if self.board.is_checkmate():
            # if it's black's turn and game over, white won
            return 1 if self.board.turn == chess.BLACK else -1
        elif self.board.is_stalemate() or self.board.is_insufficient_material():
            return 0
        else:
            return self.evaluate_board()
    
    def evaluate_board(self):
        values = {
            chess.PAWN: 1,
            chess.KNIGHT: 3,
            chess.BISHOP: 3,
            chess.ROOK: 5,
            chess.QUEEN: 9,
            chess.KING: 0,
        }
        material_score = sum(
            (values[piece.piece_type] if piece.color == chess.WHITE else -values[piece.piece_type])
            for piece in self.board.piece_map().values()
        )
        return material_score
    
# MCTS Tree Node
class Node:
    def __init__(self, state, parent=None):
        self.state = state         # GameState object
        self.parent = parent
        self.children = []
        self.visits = 0
        self.wins = 0
        self.untried_moves = state.get_possible_moves()
        self.player_turn = state.board.turn
        self.is_terminal = state.is_terminal()
    
    def is_fully_expanded(self):
        return len(self.children) == len(self.state.get_possible_moves())
    
    def best_child(self, c_param=1.4):
        choices_weights = [
            (child.wins / (child.visits + 1e-8)) + c_param * math.sqrt((2 * math.log(self.visits + 1) / (child.visits + 1e-8)))
            for child in self.children
        ]
        return self.children[choices_weights.index(max(choices_weights))]

# MCTS Class
class MCTS:
    def __init__(self, mcts_iterations=100):
        self.mcts_iterations = mcts_iterations
    
    def search(self, initial_state):
        root = Node(initial_state)
        for iteration in range(self.mcts_iterations):
            c_param = self.dynamic_c_param(iteration)
            node = self.select(root, c_param)
            reward = self.simulate(node.state)
            self.backpropagate(node, reward)
        return root.best_child(0).state  # c_param=0 => pure exploitation

    def dynamic_c_param(self, iteration):
        max_c_param, min_c_param = 1.4, 0.1
        return max(min_c_param, max_c_param * (1 - iteration / self.mcts_iterations))

    def select(self, node, c_param):
        while not node.state.is_terminal():
            if not node.is_fully_expanded():
                return self.expand(node)
            else:
                node = node.best_child(c_param)
        return node

    def expand(self, node):
        tried_moves = [child.state.board.peek() for child in node.children if node.children]
        untried_moves = [move for move in node.state.get_possible_moves() if move not in tried_moves]
        move = random.choice(untried_moves)
        new_state = node.state.make_move(move)
        child_node = Node(new_state, node)
        node.children.append(child_node)
        return child_node

    def simulate(self, state):
        current_state = state
        while not current_state.is_terminal():
            possible_moves = current_state.get_possible_moves()
            move = random.choice(possible_moves)
            current_state = current_state.make_move(move)
        return current_state.get_reward()

    def backpropagate(self, node, reward):
        while node:
            node.visits += 1
            node.wins += reward
            node = node.parent

    def simulate_multiple_moves_parallel(self, game_state, mcts_iterations, depth=3):
        legal_moves = list(game_state.board.legal_moves)
        args = [(move, game_state, mcts_iterations, depth) for move in legal_moves]
        with ThreadPool() as pool:
            results = pool.map(simulate_single_move, args)
        return results

# Simplified simulate_single_move function
def simulate_single_move(args):
    move, game_state, mcts_iterations, depth = args
    sequence = [move.uci()]
    try:
        current_state = game_state.make_move(move)
    except ValueError as e:
        return {
            "move": move.uci(),
            "sequence": [],
            "reward": None,
            "is_terminal": False,
            "board": game_state.board.fen(),
            "error": str(e)
        }
    mcts = MCTS(mcts_iterations)
    for _ in range(depth - 1):
        if current_state.is_terminal():
            break
        best_state = mcts.search(current_state)
        best_move = best_state.board.peek()
        current_state = best_state
        sequence.append(best_move.uci())
    return {
        "move": move.uci(),
        "sequence": sequence,
        "reward": current_state.get_reward(),
        "is_terminal": current_state.is_terminal(),
        "board": current_state.board.fen(),
    }

In [3]:
if __name__ == "__main__":
    # Initialize the game state
    game_state = GameState()
    mcts_iterations = 10
    mcts = MCTS()

    # Simulate multiple moves in parallel
    results = mcts.simulate_multiple_moves_parallel(game_state, mcts_iterations, depth=3)

    # Print the consequences of each move
    for result in results:
        print(f"Move: {result['move']}")
        print(f"  Sequence: {' -> '.join(result['sequence'])}")
        print(f"  Reward: {result['reward']}")
        print(f"  Is Terminal: {result['is_terminal']}")
        print()

Move: g1h3
  Sequence: g1h3 -> f7f6 -> h3g1
  Reward: 0
  Is Terminal: False

Move: g1f3
  Sequence: g1f3 -> a7a5 -> h1g1
  Reward: 0
  Is Terminal: False

Move: b1c3
  Sequence: b1c3 -> f7f5 -> g1h3
  Reward: 0
  Is Terminal: False

Move: b1a3
  Sequence: b1a3 -> c7c5 -> d2d4
  Reward: 0
  Is Terminal: False

Move: h2h3
  Sequence: h2h3 -> b8a6 -> h3h4
  Reward: 0
  Is Terminal: False

Move: g2g3
  Sequence: g2g3 -> g7g5 -> e2e3
  Reward: 0
  Is Terminal: False

Move: f2f3
  Sequence: f2f3 -> a7a6 -> a2a3
  Reward: 0
  Is Terminal: False

Move: e2e3
  Sequence: e2e3 -> c7c6 -> h2h3
  Reward: 0
  Is Terminal: False

Move: d2d3
  Sequence: d2d3 -> c7c5 -> f2f4
  Reward: 0
  Is Terminal: False

Move: c2c3
  Sequence: c2c3 -> e7e6 -> b2b4
  Reward: 0
  Is Terminal: False

Move: b2b3
  Sequence: b2b3 -> b7b6 -> b1c3
  Reward: 0
  Is Terminal: False

Move: a2a3
  Sequence: a2a3 -> h7h6 -> g2g4
  Reward: 0
  Is Terminal: False

Move: h2h4
  Sequence: h2h4 -> d7d6 -> d2d3
  Reward: 0
  Is Ter

In [4]:
if __name__ == "__main__":
    # Initialize the game state
    game_state = GameState()
    mcts_iterations = 10
    mcts = MCTS()

    # Simulate multiple moves in parallel
    results = mcts.simulate_multiple_moves_parallel(game_state, mcts_iterations, depth=5)

    # Print the consequences of each move
    for result in results:
        print(f"Move: {result['move']}")
        print(f"  Sequence: {' -> '.join(result['sequence'])}")
        print(f"  Reward: {result['reward']}")
        print(f"  Is Terminal: {result['is_terminal']}")
        print()

Move: g1h3
  Sequence: g1h3 -> g8h6 -> g2g3 -> e7e5 -> e2e3
  Reward: 0
  Is Terminal: False

Move: g1f3
  Sequence: g1f3 -> a7a6 -> g2g3 -> g8f6 -> e2e4
  Reward: 0
  Is Terminal: False

Move: b1c3
  Sequence: b1c3 -> f7f6 -> a1b1 -> d7d6 -> c3d5
  Reward: 0
  Is Terminal: False

Move: b1a3
  Sequence: b1a3 -> g8h6 -> b2b3 -> f7f5 -> a3b1
  Reward: 0
  Is Terminal: False

Move: h2h3
  Sequence: h2h3 -> g7g6 -> d2d4 -> a7a5 -> g2g4
  Reward: 0
  Is Terminal: False

Move: g2g3
  Sequence: g2g3 -> b7b5 -> f2f3 -> e7e5 -> e2e4
  Reward: 0
  Is Terminal: False

Move: f2f3
  Sequence: f2f3 -> h7h5 -> b2b3 -> g7g6 -> g2g4
  Reward: 0
  Is Terminal: False

Move: e2e3
  Sequence: e2e3 -> g7g5 -> g2g4 -> e7e5 -> h2h3
  Reward: 0
  Is Terminal: False

Move: d2d3
  Sequence: d2d3 -> g8f6 -> b1c3 -> f6g4 -> h2h4
  Reward: 0
  Is Terminal: False

Move: c2c3
  Sequence: c2c3 -> h7h6 -> d2d3 -> c7c6 -> f2f4
  Reward: 0
  Is Terminal: False

Move: b2b3
  Sequence: b2b3 -> e7e6 -> f2f3 -> d8h4 -> g2g3


In [5]:
if __name__ == "__main__":
    # Initialize the game state
    game_state = GameState()
    mcts_iterations = 10
    mcts = MCTS()

    # Simulate multiple moves in parallel
    results = mcts.simulate_multiple_moves_parallel(game_state, mcts_iterations, depth=10)

    # Print the consequences of each move
    for result in results:
        print(f"Move: {result['move']}")
        print(f"  Sequence: {' -> '.join(result['sequence'])}")
        print(f"  Reward: {result['reward']}")
        print(f"  Is Terminal: {result['is_terminal']}")
        print()

Move: g1h3
  Sequence: g1h3 -> g7g6 -> h3g5 -> f7f5 -> f2f3 -> f8h6 -> c2c4 -> h6f8 -> c4c5 -> d7d5
  Reward: 0
  Is Terminal: False

Move: g1f3
  Sequence: g1f3 -> d7d6 -> f3g1 -> h7h5 -> e2e4 -> c8d7 -> d1g4 -> b8c6 -> g4e2 -> d7f5
  Reward: 0
  Is Terminal: False

Move: b1c3
  Sequence: b1c3 -> b7b6 -> b2b3 -> b8a6 -> a1b1 -> e7e6 -> b1a1 -> a8b8 -> c1a3 -> b8a8
  Reward: 0
  Is Terminal: False

Move: b1a3
  Sequence: b1a3 -> e7e5 -> f2f3 -> f8e7 -> e2e4 -> h7h6 -> f1e2 -> a7a5 -> e2b5 -> b7b6
  Reward: 0
  Is Terminal: False

Move: h2h3
  Sequence: h2h3 -> g7g5 -> d2d4 -> e7e6 -> a2a3 -> g8e7 -> d1d2 -> f7f5 -> d2a5 -> b8c6
  Reward: 0
  Is Terminal: False

Move: g2g3
  Sequence: g2g3 -> b8a6 -> f2f4 -> d7d5 -> b2b4 -> f7f6 -> h2h3 -> c8h3 -> e2e3 -> h7h5
  Reward: -1
  Is Terminal: False

Move: f2f3
  Sequence: f2f3 -> e7e5 -> d2d3 -> c7c6 -> e1d2 -> g7g6 -> b1c3 -> f7f5 -> g2g4 -> g8f6
  Reward: 0
  Is Terminal: False

Move: e2e3
  Sequence: e2e3 -> g8f6 -> d1f3 -> b8c6 -> d2d3 -